# Fantasy Football Data Pros API Ingestion

✅ **STATUS: HISTORICAL DATA (1999-2020)** ⚠️ **NO RECENT DATA (2021+)**

Pull weekly NFL stats from Fantasy Football Data Pros free REST API and land in bronze/silver Delta tables.

**Data Coverage:**
- ✅ **22 seasons available**: 1999-2020 (562-725 players per season)
- ❌ **Missing recent seasons**: 2021, 2022, 2023, 2024, 2025
- ⚠️ API last updated in 2020, not actively maintained for current seasons

**Use Cases:**
- ✅ **Historical backtesting** - Perfect for training ML models on 22 years of data
- ✅ **Historical similarity search** - Find comparable player profiles from 1999-2020
- ❌ **Current season analysis** - Use nflverse or NextGenStats instead

**Features:**
- Free REST API with no authentication required
- Season-aggregated stats (not per-week with /all endpoint)
- Nested stats structure (passing, rushing, receiving)
- No rate limits documented

**Resources:**
- Website: https://www.fantasyfootballdatapros.com/our_api
- Working endpoint: https://www.fantasyfootballdatapros.com/api/players/{1999-2020}/all

In [0]:
import requests
import pandas as pd
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Configuration
BASE_URL = "https://www.fantasyfootballdatapros.com"  # Fixed: changed from api. subdomain to www.
SEASON = 2020  # Testing with known working season first
WEEK = "all"  # Fetch all weeks

print(f"📅 Fetching Fantasy Football Data Pros data for Season {SEASON}, Week: {WEEK}")
print(f"\nAPI Endpoint: {BASE_URL}")
print(f"\nAvailable endpoints:")
print("  - /api/players/{season}/all - All weekly stats for a season")
print("  - /api/players/{season}/{week} - Stats for specific week")

In [0]:
# Fetch weekly player statistics
print("Fetching weekly player stats from Fantasy Football Data Pros...")

try:
    # Construct API URL - try specific week first, fall back to all if needed
    url = f"{BASE_URL}/api/players/{SEASON}/{WEEK}"
    
    print(f"API URL: {url}")
    
    # Make GET request (no authentication required)
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    
    # Parse JSON response
    players_data = response.json()
    
    print(f"\nResponse type: {type(players_data)}")
    print(f"Response keys (if dict): {players_data.keys() if isinstance(players_data, dict) else 'N/A'}")
    
    # Handle different response formats
    if isinstance(players_data, dict):
        # If response is a dict, extract the player list
        # Common patterns: {'players': [...]} or {'data': [...]}
        if 'players' in players_data:
            players_list = players_data['players']
        elif 'data' in players_data:
            players_list = players_data['data']
        else:
            # If it's a single player object, wrap in list
            players_list = [players_data]
    elif isinstance(players_data, list):
        players_list = players_data
    else:
        raise ValueError(f"Unexpected response type: {type(players_data)}")
    
    print(f"✓ Fetched {len(players_list)} player records")
    
    # Convert to pandas DataFrame for easier manipulation
    players_df = pd.DataFrame(players_list)
    
    print(f"\nAvailable columns: {list(players_df.columns)}")
    
    # Display sample data
    if len(players_df) > 0:
        print(f"\nSample data:")
        display(players_df.head(5))
    
except requests.exceptions.RequestException as e:
    print(f"❌ Error fetching data: {e}")
    raise
except json.JSONDecodeError as e:
    print(f"❌ Error parsing JSON response: {e}")
    raise

In [0]:
# Test which seasons have data available (1999-2025 per documentation)
print("Testing available seasons (1999-2025)...\n")

available_seasons = []
test_years = list(range(2025, 1998, -1))  # 2025 down to 1999

for year in test_years:
    url = f"{BASE_URL}/api/players/{year}/all"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if isinstance(data, dict) and 'msg' in data:
            print(f"❌ {year}: {data['msg']}")
        elif isinstance(data, list) and len(data) > 0:
            print(f"✓ {year}: {len(data)} players available")
            available_seasons.append(year)
        else:
            print(f"⚠️  {year}: Unknown response format")
    except json.JSONDecodeError as e:
        print(f"❌ {year}: JSON decode error (empty or malformed response)")
    except Exception as e:
        print(f"❌ {year}: Error - {type(e).__name__}")

print(f"\n✓ Available seasons: {available_seasons}")
print(f"Total available: {len(available_seasons)} seasons")

In [0]:
# Batch ingest all 22 historical seasons for ML training
import time
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

print("🚀 Starting batch ingestion of 22 historical seasons (1999-2020)")
print("="*70)

# Define schema matching the target table (player_id, week, season, fantasy_points, stats, source, ingested_at)
schema = StructType([
    StructField("player_id", StringType(), False),
    StructField("week", IntegerType(), True),
    StructField("season", IntegerType(), False),
    StructField("fantasy_points", DoubleType(), True),
    StructField("stats", StringType(), True),
    StructField("source", StringType(), True),
    StructField("ingested_at", TimestampType(), True)
])

# Track statistics
total_players = 0
successful_seasons = []
failed_seasons = []
season_stats = []

# All available seasons (from our test)
available_seasons = [2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011, 
                      2010, 2009, 2008, 2007, 2006, 2005, 2004, 2003, 2002, 2001, 2000, 1999]

for season in available_seasons:
    try:
        print(f"\n📅 Processing Season {season}...")
        start_time = time.time()
        
        # Fetch data for this season
        url = f"{BASE_URL}/api/players/{season}/all"
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        players_data = response.json()
        
        if not isinstance(players_data, list) or len(players_data) == 0:
            print(f"  ⚠️  No data returned for {season}")
            failed_seasons.append(season)
            continue
        
        print(f"  ✓ Fetched {len(players_data)} players")
        
        # Convert to pandas DataFrame
        players_df = pd.DataFrame(players_data)
        
        # Transform to tuples for explicit schema
        rows = []
        for idx, row in players_df.iterrows():
            # Handle nested stats structure
            stats_dict = row.get('stats', {})
            passing = stats_dict.get('passing', {}) if isinstance(stats_dict, dict) else {}
            rushing = stats_dict.get('rushing', {}) if isinstance(stats_dict, dict) else {}
            receiving = stats_dict.get('receiving', {}) if isinstance(stats_dict, dict) else {}
            
            # Extract stats with safe defaults
            passing_yards = float(passing.get('passing_yds', 0) or 0)
            passing_tds = float(passing.get('passing_td', 0) or 0)
            interceptions = float(passing.get('int', 0) or 0)
            rushing_yards = float(rushing.get('rushing_yds', 0) or 0)
            rushing_tds = float(rushing.get('rushing_td', 0) or 0)
            receptions = float(receiving.get('receptions', 0) or 0)
            receiving_yards = float(receiving.get('receiving_yds', 0) or 0)
            receiving_tds = float(receiving.get('receiving_td', 0) or 0)
            fumbles_lost = float(row.get('fumbles_lost', 0) or 0)
            
            # Calculate PPR fantasy points
            fantasy_points = (
                (passing_yards * 0.04) +
                (passing_tds * 4) -
                (interceptions * 1) +
                (rushing_yards * 0.1) +
                (rushing_tds * 6) +
                (receptions * 1) +
                (receiving_yards * 0.1) +
                (receiving_tds * 6) -
                (fumbles_lost * 2)
            )
            
            # Create player_id
            player_name = str(row.get('player_name', ''))
            player_id = f"{player_name.lower().replace(' ', '_')}_{season}"
            
            # Store all stats as JSON (matching existing table schema)
            stats_json = json.dumps({
                "player_name": player_name,
                "position": str(row.get('position', '')),
                "team": str(row.get('team', '')),
                "games_played": int(row.get('games_played', 0) or 0),
                "passing_yards": passing_yards,
                "passing_tds": passing_tds,
                "interceptions": interceptions,
                "rushing_yards": rushing_yards,
                "rushing_tds": rushing_tds,
                "receptions": receptions,
                "receiving_yards": receiving_yards,
                "receiving_tds": receiving_tds,
                "fumbles_lost": fumbles_lost,
                "raw_data": row.to_dict()
            }, default=str)
            
            rows.append((
                player_id,
                None,  # week (season-aggregated)
                season,
                fantasy_points,
                stats_json,
                "fantasy_data_pros_historical",
                datetime.now()
            ))
        
        # Create Spark DataFrame with explicit schema
        season_df = spark.createDataFrame(rows, schema)
        
        # Write to bronze table
        season_df.createOrReplaceTempView(f"temp_bronze_{season}")
        
        spark.sql(f"""
          MERGE INTO main.fantasai.bronze_weekly_stats AS target
          USING temp_bronze_{season} AS source
          ON target.player_id = source.player_id 
            AND target.season = source.season
            AND target.source = source.source
          WHEN MATCHED THEN
            UPDATE SET *
          WHEN NOT MATCHED THEN
            INSERT *
        """)
        
        # Write to silver table (filtered for quality)
        silver_df = season_df.filter(F.col("fantasy_points") > 0).dropDuplicates(["player_id", "season"])
        silver_df.createOrReplaceTempView(f"temp_silver_{season}")
        
        spark.sql(f"""
          MERGE INTO main.fantasai.silver_weekly_stats AS target
          USING temp_silver_{season} AS source
          ON target.player_id = source.player_id 
            AND target.season = source.season
            AND target.source = source.source
          WHEN MATCHED THEN
            UPDATE SET *
          WHEN NOT MATCHED THEN
            INSERT *
        """)
        
        elapsed = time.time() - start_time
        player_count = len(rows)
        total_players += player_count
        successful_seasons.append(season)
        season_stats.append({"season": season, "players": player_count, "time": elapsed})
        
        print(f"  ✓ Wrote {player_count} players to bronze/silver tables ({elapsed:.1f}s)")
        
    except Exception as e:
        print(f"  ❌ Error processing {season}: {e}")
        failed_seasons.append(season)
        continue

# Summary
print("\n" + "="*70)
print("📊 BATCH INGESTION COMPLETE")
print("="*70)
print(f"✓ Successful seasons: {len(successful_seasons)}/22")
print(f"✓ Total players ingested: {total_players:,}")
print(f"❌ Failed seasons: {len(failed_seasons)}")
if failed_seasons:
    print(f"   Failed: {failed_seasons}")

print("\n📈 Season-by-Season Breakdown:")
for stat in season_stats:
    print(f"   {stat['season']}: {stat['players']:>4} players ({stat['time']:.1f}s)")

print(f"\n🎯 Data ready for ML training: {total_players:,} player-seasons (1999-2020)")
print("   Tables: main.fantasai.bronze_weekly_stats, main.fantasai.silver_weekly_stats")

In [0]:
%sql
-- Verify historical data from Fantasy Football Data Pros
SELECT 
  source,
  season,
  COUNT(DISTINCT player_id) as unique_players,
  COUNT(*) as total_records,
  ROUND(AVG(fantasy_points), 2) as avg_fantasy_points,
  ROUND(MAX(fantasy_points), 2) as max_fantasy_points
FROM main.fantasai.silver_weekly_stats
WHERE source = 'fantasy_data_pros_historical'
GROUP BY source, season
ORDER BY season DESC

In [0]:
%sql
-- Step 1: Add new columns to silver_weekly_stats (existing data preserved)
-- This is a SAFE operation - no data loss

ALTER TABLE main.fantasai.silver_weekly_stats 
ADD COLUMNS (
  player_name STRING,
  position STRING,
  team STRING,
  games_played INT,
  passing_yards DOUBLE,
  passing_tds DOUBLE,
  interceptions DOUBLE,
  rushing_yards DOUBLE,
  rushing_tds DOUBLE,
  receptions DOUBLE,
  receiving_yards DOUBLE,
  receiving_tds DOUBLE,
  fumbles_lost DOUBLE
);

-- Verify table schema after adding columns
DESCRIBE TABLE main.fantasai.silver_weekly_stats;

In [0]:
%sql
-- Step 2: Backfill the new columns from existing JSON stats field
-- FIXED: Using correct nested JSON paths based on actual data structure
-- The stats are stored as: $.stats.passing.passing_yds (nested), not $.passing_yards (flat)

UPDATE main.fantasai.silver_weekly_stats
SET 
  -- Top-level fields (these work)
  player_name = COALESCE(player_name, get_json_object(stats, '$.player_name')),
  position = COALESCE(position, get_json_object(stats, '$.position')),
  team = COALESCE(team, get_json_object(stats, '$.team')),
  games_played = COALESCE(games_played, CAST(CAST(get_json_object(stats, '$.games_played') AS DOUBLE) AS INT)),
  fumbles_lost = COALESCE(fumbles_lost, CAST(get_json_object(stats, '$.fumbles_lost') AS DOUBLE)),
  -- Nested stats fields (these need nested paths)
  passing_yards = COALESCE(passing_yards, CAST(get_json_object(stats, '$.stats.passing.passing_yds') AS DOUBLE)),
  passing_tds = COALESCE(passing_tds, CAST(get_json_object(stats, '$.stats.passing.passing_td') AS DOUBLE)),
  interceptions = COALESCE(interceptions, CAST(get_json_object(stats, '$.stats.passing.int') AS DOUBLE)),
  rushing_yards = COALESCE(rushing_yards, CAST(get_json_object(stats, '$.stats.rushing.rushing_yds') AS DOUBLE)),
  rushing_tds = COALESCE(rushing_tds, CAST(get_json_object(stats, '$.stats.rushing.rushing_td') AS DOUBLE)),
  receptions = COALESCE(receptions, CAST(get_json_object(stats, '$.stats.receiving.receptions') AS DOUBLE)),
  receiving_yards = COALESCE(receiving_yards, CAST(get_json_object(stats, '$.stats.receiving.receiving_yds') AS DOUBLE)),
  receiving_tds = COALESCE(receiving_tds, CAST(get_json_object(stats, '$.stats.receiving.receiving_td') AS DOUBLE))
WHERE source = 'fantasy_data_pros_historical';

-- Verify backfill worked
SELECT 
  COUNT(*) as total_rows,
  COUNT(player_name) as has_player_name,
  COUNT(passing_yards) as has_passing_yards,
  COUNT(rushing_yards) as has_rushing_yards,
  COUNT(receiving_yards) as has_receiving_yards
FROM main.fantasai.silver_weekly_stats
WHERE source = 'fantasy_data_pros_historical';

In [0]:
%sql
-- Top 20 fantasy performers from 2020 (most recent historical season)
-- Now using the new columns directly (no JSON parsing needed!)
SELECT 
  player_name,
  position,
  team,
  season,
  ROUND(fantasy_points, 2) as fantasy_points,
  ROUND(passing_yards, 0) as pass_yds,
  ROUND(passing_tds, 0) as pass_tds,
  ROUND(rushing_yards, 0) as rush_yds,
  ROUND(rushing_tds, 0) as rush_tds,
  ROUND(receptions, 0) as rec,
  ROUND(receiving_yards, 0) as rec_yds,
  ROUND(receiving_tds, 0) as rec_tds,
  games_played
FROM main.fantasai.silver_weekly_stats
WHERE source = 'fantasy_data_pros_historical' AND season = 2020
ORDER BY fantasy_points DESC
LIMIT 20

In [0]:
%sql
-- Check what's actually in the stats JSON field
SELECT 
  player_name,
  passing_yards,
  rushing_yards,
  stats
FROM main.fantasai.silver_weekly_stats
WHERE source = 'fantasy_data_pros_historical' 
  AND season = 2020
  AND player_name = 'Josh Allen'
LIMIT 1

In [0]:
%sql
-- Check both the flat and nested paths in JSON
SELECT 
  player_name,
  -- Try flat paths first
  get_json_object(stats, '$.passing_yards') as flat_pass_yards,
  get_json_object(stats, '$.rushing_yards') as flat_rush_yards,
  -- Try nested paths  
  get_json_object(stats, '$.raw_data.stats.passing.passing_yds') as nested_pass_yards,
  get_json_object(stats, '$.raw_data.stats.rushing.rushing_yds') as nested_rush_yards
FROM main.fantasai.silver_weekly_stats
WHERE source = 'fantasy_data_pros_historical' 
  AND season = 2020
  AND player_name = 'Josh Allen'
LIMIT 1

In [0]:
# Transform pandas DataFrame to Spark DataFrame with standardized schema
if 'players_df' in locals() and len(players_df) > 0:
    print("Transforming to Spark DataFrame...")
    
    # Create rows for Spark DataFrame
    rows = []
    for idx, row in players_df.iterrows():
        # Extract key fields with fallbacks
        player_id = str(row.get('player_id', row.get('name', f'unknown_{idx}')))
        player_name = str(row.get('name', ''))
        position = str(row.get('position', ''))
        team = str(row.get('team', ''))
        
        # Calculate fantasy points (PPR scoring if not provided)
        fantasy_points = float(row.get('fantasy_points', 0) or 0)
        
        # If fantasy_points not in data, calculate from stats
        if fantasy_points == 0:
            # PPR scoring calculation
            passing_yards = float(row.get('passing_yards', 0) or 0)
            passing_tds = float(row.get('passing_touchdowns', 0) or 0)
            rushing_yards = float(row.get('rushing_yards', 0) or 0)
            rushing_tds = float(row.get('rushing_touchdowns', 0) or 0)
            receptions = float(row.get('receptions', 0) or 0)
            receiving_yards = float(row.get('receiving_yards', 0) or 0)
            receiving_tds = float(row.get('receiving_touchdowns', 0) or 0)
            fumbles_lost = float(row.get('fumbles_lost', 0) or 0)
            
            fantasy_points = (
                (passing_yards * 0.04) +
                (passing_tds * 4) +
                (rushing_yards * 0.1) +
                (rushing_tds * 6) +
                (receptions * 1) +
                (receiving_yards * 0.1) +
                (receiving_tds * 6) -
                (fumbles_lost * 2)
            )
        
        # Convert entire row to JSON for stats column
        stats_json = json.dumps(row.to_dict(), default=str)
        
        rows.append(
            Row(
                player_id=player_id,
                player_name=player_name,
                position=position,
                team=team,
                week=WEEK,
                season=SEASON,
                fantasy_points=fantasy_points,
                stats=stats_json
            )
        )
    
    # Create Spark DataFrame
    stats_df = spark.createDataFrame(rows)
    
    print(f"✓ Created Spark DataFrame with {stats_df.count()} records")
    
    # Show sample
    print("\nSample Spark DataFrame:")
    display(stats_df.limit(10))
    
else:
    print("❌ No data available to transform")

In [0]:
# Write to bronze table using MERGE
if 'stats_df' in locals():
    print("Writing to bronze_weekly_stats...")
    
    bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
    bronze_df = bronze_df.withColumn("source", F.lit("fantasy_data_pros"))
    
    # Create temp view for merge
    bronze_df.createOrReplaceTempView("fantasy_data_pros_bronze_updates")
    
    # Perform MERGE operation
    merge_result = spark.sql("""
      MERGE INTO main.fantasai.bronze_weekly_stats AS target
      USING fantasy_data_pros_bronze_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.player_name = source.player_name,
          target.position = source.position,
          target.team = source.team,
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.source = source.source,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, player_name, position, team, week, season, fantasy_points, stats, source, ingested_at)
        VALUES (source.player_id, source.player_name, source.position, source.team, source.week, source.season, 
                source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    print(f"✓ Merged {bronze_df.count()} records from Fantasy Football Data Pros into bronze_weekly_stats")
    
else:
    print("❌ No data available to write")

In [0]:
# Transform for silver table
if 'bronze_df' in locals():
    print("Transforming and writing to silver_weekly_stats...")
    
    silver_df = (
        bronze_df
        .select(
            F.col("player_id").cast("string"),
            F.col("player_name").cast("string"),
            F.col("position").cast("string"),
            F.col("team").cast("string"),
            F.col("week").cast("int"),
            F.col("season").cast("int"),
            F.col("fantasy_points").cast("double"),
            F.col("stats").cast("string"),
            F.col("source"),
            F.col("ingested_at"),
        )
        .dropDuplicates(["player_id", "week", "season"])
        .filter(F.col("fantasy_points") > 0)  # Only include players with fantasy points
    )
    
    # Create temp view for merge
    silver_df.createOrReplaceTempView("fantasy_data_pros_silver_updates")
    
    # Perform MERGE operation
    spark.sql("""
      MERGE INTO main.fantasai.silver_weekly_stats AS target
      USING fantasy_data_pros_silver_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.player_name = source.player_name,
          target.position = source.position,
          target.team = source.team,
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.source = source.source,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, player_name, position, team, week, season, fantasy_points, stats, source, ingested_at)
        VALUES (source.player_id, source.player_name, source.position, source.team, source.week, source.season,
                source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    print(f"✓ Merged {silver_df.count()} records from Fantasy Football Data Pros into silver_weekly_stats")
    
else:
    print("❌ No data available to write")

In [0]:
%sql
-- Check Fantasy Football Data Pros data in silver table
SELECT 
  player_id,
  player_name,
  position,
  team,
  week,
  season,
  fantasy_points,
  source
FROM main.fantasai.silver_weekly_stats
WHERE week = 18 AND season = 2024 AND source = 'fantasy_data_pros'
ORDER BY fantasy_points DESC
LIMIT 20

## Fantasy Football Data Pros API - Complete Assessment

### ✅ What Works
- **22 seasons of historical data** (1999-2020)
- **12,800+ player-seasons** total across all years
- Consistent JSON structure across all years
- Nested stats: passing, rushing, receiving, fumbles
- No authentication or rate limits

### ❌ Critical Limitation
- **No data after 2020** - Missing 2021, 2022, 2023, 2024, 2025 seasons
- API appears abandoned after 2020
- Documentation outdated (claims 1999+ but really 1999-2020)

### 📊 Data Available by Year
```
2020: 626 players ✓
2019: 620 players ✓
2018: 622 players ✓
2017: 571 players ✓
...
2000: 557 players ✓
1999: 562 players ✓
```

### 🎯 Recommended Use
1. **Primary use:** Historical backtesting and ML model training (1999-2020)
2. **Secondary use:** Historical similarity search for player comps
3. **DO NOT use for:** Current season predictions, waiver analysis, or 2021+ data

### 🚀 Next Steps
1. Ingest all 22 seasons (1999-2020) to bronze/silver tables
2. Test nflverse (Notebook 14) for current season data (2021-2025)
3. Build NextGenStats for routes run + air yards (current seasons)
4. Combine: Historical baseline from Fantasy Data Pros + Current trends from nflverse/NextGen